# Étape 2 — Préparation et analyse exploratoire des données (EDA)

**Objectif :** comprendre les données avant de modéliser, et préparer des features sans fuite d'information.

**Règles suivies dans ce notebook**
1. Le découpage train / validation / test est **chronologique** et fait **avant** tout prétraitement.
2. Les statistiques qui guident nos choix (corrélations, distributions, stationnarité) sont calculées sur le **train uniquement**.
3. La période de **test n'est jamais analysée** ici : elle reste intacte jusqu'à l'évaluation finale (étape 8).

Toutes les figures sont enregistrées dans `reports/figures/` pour le rapport et les slides.

In [ ]:
import sys, json
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))

import pandas as pd
from scipy import stats
from src.config import load_config
from src.data import prepare
from src import eda

cfg = load_config()
FIG = ROOT / cfg["paths"]["figures"]
PROC = ROOT / cfg["paths"]["processed"]

# Lance la préparation complète (nettoyage, découpage, features)
prepare.main()

prices = pd.read_csv(PROC / "prices_clean.csv", index_col=0, parse_dates=True)
returns = pd.read_csv(PROC / "returns.csv", index_col=0, parse_dates=True)
macro = pd.read_csv(PROC / "macro_aligned.csv", index_col=0, parse_dates=True)
report = json.load(open(PROC / "preparation_report.json", encoding="utf-8"))

train_end, val_end = cfg["split"]["train_end"], cfg["split"]["val_end"]
prices_train, returns_train = prices.loc[:train_end], returns.loc[:train_end]

## 2.1 Qualité des données

On regarde les valeurs manquantes brutes et la date de première cotation de chaque actif. La période d'étude commence à la date où **tous** les actifs sont cotés.

In [ ]:
print("Valeurs manquantes brutes :", report["valeurs_manquantes_brutes"])
print("Période commune :", report["debut_commun"], "->", report["fin"],
      f"({report['nb_jours']} jours, {report['nb_actifs']} actifs)")
pd.Series(report["premiere_date_par_actif"]).sort_values(ascending=False).head(5)

## 2.2 Découpage chronologique

Les 21 derniers jours du train et de la validation sont retirés (**purge**) : leur cible (rendement à 21 jours) déborderait sur la période suivante. Les ~252 premiers jours servent à « chauffer » les features glissantes (moyennes sur 1 an).

In [ ]:
pd.DataFrame(report["split"]).T

## 2.3 Évolution des prix

Vue d'ensemble sur toute la période, pour le contexte (krach du Covid en 2020, hausse des taux en 2022). Aucune décision n'est prise à partir de la partie test.

In [ ]:
eda.fig_prices(prices, [train_end, val_end], FIG, cols=["SPY", "QQQ", "TLT", "GLD", "AAPL", "XOM"]);

## 2.4 Statistiques par actif (train)

Rendement et volatilité annualisés, ratio de Sharpe, asymétrie (skewness), aplatissement (kurtosis en excès : 0 pour une loi normale) et pire perte depuis un sommet (drawdown max).

In [ ]:
table = eda.stats_table(returns_train)
table.to_csv(ROOT / "reports" / "stats_train.csv")
table

## 2.5 Distribution des rendements (train)

Les modèles classiques (Markowitz) supposent souvent des rendements normaux. On vérifie visuellement et avec le test de Jarque-Bera (p < 0,05 → on rejette la normalité).

In [ ]:
eda.fig_distribution(returns_train, "SPY", FIG);
jb = stats.jarque_bera(returns_train["SPY"].dropna())
print(f"Jarque-Bera SPY : statistique = {jb.statistic:.0f}, p-value = {jb.pvalue:.2e}")

## 2.6 Corrélations (train)

Les actifs sont regroupés par similarité. La diversification vient des actifs peu ou négativement corrélés (obligations TLT / IEF, or GLD).

In [ ]:
eda.fig_correlation(returns_train, FIG);

## 2.7 Volatilité glissante : les régimes de marché

La volatilité n'est pas constante : elle grimpe par épisodes (crises). C'est ce qui justifie un agent Risque qui adapte l'allocation au régime.

In [ ]:
eda.fig_rolling_vol(returns, ["SPY", "TLT", "GLD", "NVDA"], FIG);

## 2.8 Variables macroéconomiques

Taux à 10 ans, VIX (indice de peur) et taux directeur, décalés d'un jour pour ne jamais utiliser une donnée pas encore publiée.

In [ ]:
eda.fig_macro(macro, FIG);

## 2.9 Stationnarité (test de Dickey-Fuller augmenté, train)

Une série stationnaire garde les mêmes propriétés dans le temps, ce dont les modèles ont besoin. Attendu : les prix ne sont pas stationnaires (p > 0,05), les rendements le sont (p < 0,05). C'est pourquoi on modélise des **rendements**, pas des prix.

In [ ]:
adf = eda.adf_table(prices_train, returns_train)
adf.to_csv(ROOT / "reports" / "adf_train.csv")
adf

## 2.10 Aperçu des features (train)

Une ligne par (date, actif). `target` = rendement log des 21 prochains jours de bourse : c'est ce que l'agent Prévision devra prédire.

In [ ]:
feat_train = pd.read_csv(PROC / "features_train.csv", index_col=[0, 1], parse_dates=[0])
print(feat_train.shape)
feat_train.describe().T.round(3)

## 2.11 Conclusions à reporter dans le rapport

À compléter après exécution, à partir des résultats ci-dessus :

- **Qualité :** période commune du … au …, … valeurs manquantes traitées.
- **Rendements :** queues épaisses (kurtosis en excès de … pour SPY), normalité rejetée → justifie des mesures de risque comme la CVaR.
- **Corrélations :** actions très corrélées entre elles (≈ …), obligations et or peu corrélés → sources de diversification.
- **Régimes :** pics de volatilité en … → intérêt de l'agent Risque.
- **Stationnarité :** prix non stationnaires, rendements stationnaires → on modélise les rendements.